<a href="https://colab.research.google.com/github/GUNAPILLCO/neural_profit/blob/main/stage_06_model_training/stage_06_00_model_training_plan.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Stage_06 – Model Training – Objetivo y Modelos**


# **1. Plan de entrenamiento**

En este notebook se definirá el plan de entrenamiento para los modelos aplicados al problema **T2 de clasificación secuencial**, manteniendo un esquema experimental consistente, comparable y alineado con el flujo metodológico del libro: definición del problema supervisado, diseño del modelo, evaluación fuera de muestra y control del overfitting.  

## **Targets**

En esta etapa ya no se trabajará con targets continuos de regresión.
El enfoque pasa a ser exclusivamente de **clasificación T2**, con los siguientes targets:

* `t2_dir_thr_90`
* `t2_dir_thr_120`

Estos targets representan una clasificación direccional con umbral, definida a partir del movimiento futuro del precio a horizontes de 90 y 120 minutos.

## **Window size**

El modelado se realizará en formato secuencial, utilizando ventanas históricas de longitud fija como contexto de entrada.

Cada ventana contendrá una secuencia temporal de observaciones pasadas del mercado y su longitud será tratada como hiperparámetro experimental.
Los tamaños de ventana a evaluar se definirán dentro del stage según el costo computacional y la robustez observada.

## **Features**

A diferencia del planteamiento anterior, ya no se utilizarán 36 variables.
El entrenamiento se realizará con el conjunto final reducido de **5 features**, seleccionado a partir del análisis previo de señal, robustez y redundancia:

```python
features_t2 = [
    "regime_id",
    "roc_30",
    "roc_60",
    "stoch_k_30",
    "atr_norm_10",
]
```

Este conjunto busca mantener una representación compacta y con señal útil, reduciendo complejidad y riesgo de sobreajuste, especialmente importante en datos financieros con baja relación señal-ruido.

## **Enfoque de predicción**

En esta etapa el enfoque será únicamente:

* **seq2one**: una secuencia de entrada produce una única predicción.

Esto implica que cada ventana histórica será utilizada para predecir una sola etiqueta futura del target T2.
No se trabajará con enfoque **seq2seq** en este stage.

Esta decisión también está alineada con la formulación many-to-one típica para tareas secuenciales de clasificación, donde una secuencia resume el contexto y se asigna a una salida categórica única.

## **Tipo de problema**

El problema queda formalmente definido como:

* **aprendizaje supervisado**
* **clasificación multiclase**
* **modelado secuencial seq2one**
* **predicción intradía sobre datos minuto a minuto del MNQ**

## **Objetivo**

El objetivo de este stage será comparar de forma sistemática el desempeño de distintos modelos de clasificación secuencial sobre los targets T2, analizando:

* diferencias entre `t2_dir_thr_90` y `t2_dir_thr_120`
* sensibilidad al tamaño de ventana
* capacidad predictiva de distintas arquitecturas seq2one
* robustez fuera de muestra

El propósito no es solo obtener el mejor score puntual, sino identificar un esquema de modelado que sea consistente, interpretable y metodológicamente sólido para etapas posteriores de validación.


# **2. Estructura dimensional del problema de predicción**

## **2.1. Ventana de entrada $X$**

Para ambos enfoques de predicción, las ventanas de entrada $X$ tienen la siguiente dimensión:

$$
L \times N
$$

donde:

- $L$ representa la longitud de la ventana histórica y puede tomar los valores $[30, 60, 90, 120, 180]$.
- $N$ corresponde al número de variables predictoras (features), que en todos los casos es igual a **5**.


## **2.2 Ventana de salida $y$**

### **2.2.1. Enfoque `seq2one`**


En este esquema se entrenarán modelos **seq2one** para predecir un **único valor escalar** asociado a cada ventana histórica.

- **Salida / Target (Y)**

  $$
  Y \in \mathbb{R}^{1}
  $$

  Este valor representa la variable objetivo futura definida en los stages previos.

- **Interpretación**

  El modelo aprende una función del tipo:

  $$
  f: \mathbb{R}^{L \times N} \rightarrow \mathbb{R}
  $$

  Es decir, a partir de una ventana histórica de tamaño $L \times N$, el modelo produce **una única predicción final**.

  Este enfoque es adecuado cuando el objetivo es predecir el retorno o delta acumulado a un horizonte fijo $H$.

# **2. Modelos a entrenar**


En este stage se evaluarán distintas familias de modelos para el problema de **clasificación secuencial seq2one (T2)**, buscando un balance entre:

* robustez en datos financieros (ruidosos)
* capacidad de capturar dependencias temporales
* complejidad vs generalización

## **2.1 Enfoque general**

Se trabajará con dos grandes enfoques complementarios:

### 1) Modelos tabulares (baseline)

Estos modelos **no utilizan la estructura secuencial explícita**, sino que operan sobre features agregadas o representaciones simplificadas.

Su objetivo es:

* establecer un baseline sólido
* verificar si la secuencia realmente aporta valor
* reducir riesgo de sobreajuste

Esto es consistente con el libro: los modelos simples y robustos suelen ser un punto de partida fuerte en datos financieros

Modelos incluidos:

* Logistic Regression (baseline principal)
* Random Forest
* Gradient Boosting (LightGBM / XGBoost)

Los métodos de boosting son especialmente relevantes porque suelen dominar en datos tabulares estructurados

---

### 2) Modelos secuenciales (deep learning)

Estos modelos sí explotan explícitamente la dimensión temporal de los datos.

Se utilizarán arquitecturas diseñadas para secuencias, donde la predicción depende del contexto histórico.

Esto es clave porque:

* los datos financieros son series temporales
* existe dependencia temporal (aunque débil y ruidosa)
* algunas arquitecturas pueden capturar patrones dinámicos

Modelos incluidos:

#### a) LSTM

* Captura dependencias temporales mediante memoria interna
* Adecuado para secuencias financieras
* Buen baseline dentro de deep learning

#### b) GRU

* Variante más simple que LSTM
* Menor complejidad → menor riesgo de overfitting
* Más eficiente computacionalmente

#### c) Transformer (Encoder)

* Modela relaciones globales en la secuencia
* No depende de recurrencia
* Ha mostrado buen desempeño en tu etapa previa

---

## **2.2 Justificación de la selección**

La elección cubre distintos niveles de complejidad:

| Tipo de modelo     | Rol                        |
| ------------------ | -------------------------- |
| Logístico          | baseline interpretable     |
| Árboles / Boosting | baseline no lineal robusto |
| LSTM / GRU         | modelado temporal clásico  |
| Transformer        | modelado temporal avanzado |

Esto permite responder preguntas clave:

* ¿La secuencia aporta valor real?
* ¿Los modelos complejos superan a los tabulares?
* ¿Hay evidencia de señal explotable o solo ruido?

---

## **2.3 Enfoque experimental**

Todos los modelos se entrenarán bajo un esquema consistente:

* mismo split temporal (train / valid / test)
* mismas features
* mismos targets
* mismas métricas

El objetivo es garantizar comparabilidad y evitar conclusiones sesgadas, lo cual es crítico en el workflow de ML para trading

# **3. Comparación entre etiqueta real y predicción**

En este stage, la evaluación del modelo se realiza exclusivamente bajo el esquema **seq2one**.

Cada ventana histórica de longitud $L$, compuesta por $N$ features por minuto, produce una única predicción asociada a una sola etiqueta futura del target T2.

Para cada muestra $t$, la entrada del modelo es:

$$
X_t \in \mathbb{R}^{L \times N}
$$

donde:

* $L$ es la longitud de la ventana histórica
* $N$ es el número de features

El modelo produce una única predicción:

$$
\hat{y}_t \in \mathcal{C}
$$

donde $\mathcal{C}$ representa el conjunto de clases del target T2.

La predicción se compara directamente contra la etiqueta real observada:

$$
y_t \in \mathcal{C}
$$

La comparación se realiza entonces como:

* **una secuencia de entrada**
* **una única predicción**
* **una única etiqueta real asociada**

No se generan trayectorias futuras ni secuencias completas de salida.
Cada ventana histórica resume el contexto necesario para emitir una única decisión de clasificación.

## **3.1 Interpretación del esquema seq2one**

Bajo este enfoque, el modelo aprende una función del tipo:

$$
f: \mathbb{R}^{L \times N} \rightarrow \mathcal{C}
$$

es decir, transforma una ventana temporal multivariada en una única clase objetivo.

Este planteamiento es consistente con el diseño actual del problema T2, donde el interés no está en predecir toda una trayectoria futura, sino en clasificar el comportamiento futuro del precio bajo una definición discreta del target.

# **4. Métricas y evaluación del modelo**

## **4.1 Enfoque de evaluación**

Dado que el problema es de **clasificación seq2one**, cada muestra produce:

* una predicción de clase: $\hat{y}_t \in \mathcal{C}$
* una etiqueta real: $y_t \in \mathcal{C}$

Las métricas se calculan sobre el conjunto completo de muestras de cada split.

La comparación es:

* **predicción vs etiqueta real**
* no hay valores continuos
* no hay error de magnitud

---

## **4.2 Métricas de Machine Learning**

Las métricas se agrupan en tres niveles:

### **4.2.1 Métricas principales (criterio de selección)**

Estas métricas determinan qué modelos continúan:

* **Balanced Accuracy**

  * Corrige desbalance de clases
  * Métrica principal para T2

* **F1 Score (macro o weighted)**

  * Balance entre precisión y recall
  * Complementa la balanced accuracy

---

### **4.2.2 Métricas complementarias**

Se reportan para interpretación:

* **Accuracy**
* **Precision (macro / weighted)**
* **Recall (macro / weighted)**

---

### **4.2.3 Métricas diagnósticas**

Solo para análisis interno:

* matriz de confusión
* distribución de clases predichas
* comparación contra baseline (naive)

---

## **4.3 Baseline de referencia**

Todos los modelos deben compararse contra un baseline simple:

* **Naive classifier**:

  * predice siempre la clase más frecuente

Esto es clave para verificar si el modelo realmente captura señal.

---

## **4.4 Uso de los splits (criterio de evaluación)**

Se mantiene el esquema estándar del workflow:

* **TRAIN**

  * usado exclusivamente para entrenamiento

* **VALID**

  * usado para:

    * comparar modelos
    * seleccionar hiperparámetros
    * decidir qué modelos continúan

* **TEST**

  * usado solo una vez
  * evaluación final del modelo seleccionado

No se utiliza el TEST durante el entrenamiento ni tuning.

Este esquema evita leakage y garantiza evaluación fuera de muestra, lo cual es crítico en trading

---

## **4.5 Protocolo de evaluación**

Para cada modelo:

1. entrenar en TRAIN
2. evaluar en VALID
3. calcular métricas principales
4. comparar contra baseline
5. seleccionar candidatos

El TEST se utiliza únicamente al final del stage.

---

##**4.6 Métricas económicas (etapa posterior)**

Las métricas económicas no forman parte de este stage.

Una vez seleccionados los modelos candidatos, se evaluarán en el siguiente stage utilizando:

* retorno esperado (EV)
* métricas riesgo-retorno
* drawdown

Estas métricas permiten traducir el desempeño estadístico en impacto operativo, alineado con el workflow del libro (modelo → estrategia → evaluación)

---

## **4.7 Jerarquía de métricas**

**Criterio de selección:**

* Balanced Accuracy
* F1 Score

**Métricas complementarias:**

* Accuracy
* Precision
* Recall

**Diagnóstico:**

* matriz de confusión
* baseline vs modelo

# **5. Función general de métricas de clasificación**

Para estandarizar la evaluación de todos los modelos de clasificación T2, se implementa un módulo reutilizable de Python importable desde notebooks y scripts de entrenamiento.

La función principal recibe las etiquetas reales y predichas, y devuelve:

- Métricas principales de clasificación
- Métricas complementarias
- Comparación contra baseline naive
- Matriz de confusión
- Distribución de clases reales y predichas

Esto permite evaluar de forma consistente modelos tabulares y secuenciales bajo el mismo criterio experimental.

In [1]:
"""
classification_metrics.py

Utilidades de métricas para problemas de clasificación seq2one
aplicados al proyecto MNQ T2.

Diseñado para ser importado desde notebooks o scripts de entrenamiento.

Métricas principales:
- balanced_accuracy
- f1_macro
- f1_weighted

Métricas complementarias:
- accuracy
- precision_macro
- precision_weighted
- recall_macro
- recall_weighted

Diagnóstico:
- confusion_matrix
- class distribution real/predicha
- baseline naive (clase más frecuente en y_true)

Autor: OpenAI / Proyecto MNQ
"""

from __future__ import annotations

from dataclasses import dataclass
from typing import Any, Dict, Iterable, Optional

import numpy as np
from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    confusion_matrix,
    f1_score,
    precision_score,
    recall_score,
)


@dataclass
class ClassificationMetricsConfig:
    """
    Configuración para el cálculo de métricas de clasificación.
    """
    average_macro: str = "macro"
    average_weighted: str = "weighted"
    zero_division: int = 0
    include_confusion_matrix: bool = True
    include_class_distribution: bool = True
    include_naive_baseline: bool = True


def _to_1d_numpy(y: Iterable[Any], name: str) -> np.ndarray:
    """
    Convierte una entrada a np.ndarray 1D.

    Parámetros
    ----------
    y : iterable
        Etiquetas reales o predichas.
    name : str
        Nombre de la variable, solo para mensajes de error.

    Retorna
    -------
    np.ndarray
        Vector 1D.
    """
    arr = np.asarray(y)

    if arr.ndim == 0:
        raise ValueError(f"{name} no puede ser escalar; se esperaba un vector 1D.")
    if arr.ndim > 1:
        arr = arr.reshape(-1)

    if arr.size == 0:
        raise ValueError(f"{name} está vacío.")

    return arr


def _build_naive_predictions(y_true: np.ndarray) -> np.ndarray:
    """
    Construye un baseline naive que predice siempre la clase más frecuente en y_true.
    """
    classes, counts = np.unique(y_true, return_counts=True)
    majority_class = classes[np.argmax(counts)]
    return np.full_like(y_true, fill_value=majority_class)


def _compute_core_metrics(
    y_true: np.ndarray,
    y_pred: np.ndarray,
    *,
    zero_division: int = 0,
) -> Dict[str, float]:
    """
    Calcula métricas principales y complementarias.
    """
    return {
        "accuracy": float(accuracy_score(y_true, y_pred)),
        "balanced_accuracy": float(balanced_accuracy_score(y_true, y_pred)),
        "f1_macro": float(
            f1_score(y_true, y_pred, average="macro", zero_division=zero_division)
        ),
        "f1_weighted": float(
            f1_score(y_true, y_pred, average="weighted", zero_division=zero_division)
        ),
        "precision_macro": float(
            precision_score(y_true, y_pred, average="macro", zero_division=zero_division)
        ),
        "precision_weighted": float(
            precision_score(y_true, y_pred, average="weighted", zero_division=zero_division)
        ),
        "recall_macro": float(
            recall_score(y_true, y_pred, average="macro", zero_division=zero_division)
        ),
        "recall_weighted": float(
            recall_score(y_true, y_pred, average="weighted", zero_division=zero_division)
        ),
    }


def compute_classification_metrics(
    y_true: Iterable[Any],
    y_pred: Iterable[Any],
    *,
    model_name: Optional[str] = None,
    split: Optional[str] = None,
    target: Optional[str] = None,
    labels: Optional[Iterable[Any]] = None,
    config: Optional[ClassificationMetricsConfig] = None,
) -> Dict[str, Any]:
    """
    Calcula métricas estandarizadas para clasificación seq2one.

    Parámetros
    ----------
    y_true : iterable
        Etiquetas reales.
    y_pred : iterable
        Etiquetas predichas.
    model_name : str, opcional
        Nombre del modelo.
    split : str, opcional
        Split evaluado: train / valid / test.
    target : str, opcional
        Nombre del target.
    labels : iterable, opcional
        Orden explícito de clases para la matriz de confusión.
        Si es None, se usa la unión ordenada de y_true e y_pred.
    config : ClassificationMetricsConfig, opcional
        Configuración del cálculo.

    Retorna
    -------
    dict
        Diccionario con:
        - metadata
        - métricas del modelo
        - baseline naive
        - matriz de confusión
        - distribuciones de clases
    """
    cfg = config or ClassificationMetricsConfig()

    y_true_arr = _to_1d_numpy(y_true, "y_true")
    y_pred_arr = _to_1d_numpy(y_pred, "y_pred")

    if y_true_arr.shape[0] != y_pred_arr.shape[0]:
        raise ValueError(
            f"y_true e y_pred deben tener la misma longitud. "
            f"Recibido: {y_true_arr.shape[0]} vs {y_pred_arr.shape[0]}"
        )

    if labels is None:
        final_labels = np.unique(np.concatenate([y_true_arr, y_pred_arr]))
    else:
        final_labels = np.asarray(list(labels))

    metrics = {
        "model": model_name,
        "split": split,
        "target": target,
        "n_samples": int(len(y_true_arr)),
        **_compute_core_metrics(
            y_true_arr,
            y_pred_arr,
            zero_division=cfg.zero_division,
        ),
    }

    if cfg.include_naive_baseline:
        y_pred_naive = _build_naive_predictions(y_true_arr)
        naive_metrics = _compute_core_metrics(
            y_true_arr,
            y_pred_naive,
            zero_division=cfg.zero_division,
        )
        metrics.update({
            "accuracy_naive": naive_metrics["accuracy"],
            "balanced_accuracy_naive": naive_metrics["balanced_accuracy"],
            "f1_macro_naive": naive_metrics["f1_macro"],
            "f1_weighted_naive": naive_metrics["f1_weighted"],
            "balanced_accuracy_gain_vs_naive": (
                metrics["balanced_accuracy"] - naive_metrics["balanced_accuracy"]
            ),
            "f1_macro_gain_vs_naive": (
                metrics["f1_macro"] - naive_metrics["f1_macro"]
            ),
            "f1_weighted_gain_vs_naive": (
                metrics["f1_weighted"] - naive_metrics["f1_weighted"]
            ),
        })

    result: Dict[str, Any] = {"metrics": metrics}

    if cfg.include_confusion_matrix:
        cm = confusion_matrix(y_true_arr, y_pred_arr, labels=final_labels)
        result["confusion_matrix"] = cm.tolist()
        result["confusion_matrix_labels"] = final_labels.tolist()

    if cfg.include_class_distribution:
        true_classes, true_counts = np.unique(y_true_arr, return_counts=True)
        pred_classes, pred_counts = np.unique(y_pred_arr, return_counts=True)

        result["class_distribution_true"] = {
            str(cls): int(cnt) for cls, cnt in zip(true_classes, true_counts)
        }
        result["class_distribution_pred"] = {
            str(cls): int(cnt) for cls, cnt in zip(pred_classes, pred_counts)
        }

    return result


def metrics_to_flat_dict(metrics_result: Dict[str, Any]) -> Dict[str, Any]:
    """
    Aplana la salida de compute_classification_metrics para convertirla
    fácilmente en DataFrame.

    Parámetros
    ----------
    metrics_result : dict
        Salida de compute_classification_metrics.

    Retorna
    -------
    dict
        Diccionario plano solo con métricas y metadata.
    """
    if "metrics" not in metrics_result:
        raise ValueError("El diccionario recibido no contiene la clave 'metrics'.")

    return dict(metrics_result["metrics"])


def print_classification_report_block(metrics_result: Dict[str, Any]) -> None:
    """
    Imprime un bloque compacto y legible con las métricas principales.
    """
    m = metrics_result["metrics"]

    title = (
        f"CLASSIFICATION REPORT | "
        f"model={m.get('model')} | split={m.get('split')} | target={m.get('target')}"
    )
    print("=" * len(title))
    print(title)
    print("=" * len(title))
    print(f"n_samples               : {m.get('n_samples')}")
    print(f"balanced_accuracy       : {m.get('balanced_accuracy'):.6f}")
    print(f"f1_macro                : {m.get('f1_macro'):.6f}")
    print(f"f1_weighted             : {m.get('f1_weighted'):.6f}")
    print(f"accuracy                : {m.get('accuracy'):.6f}")
    print(f"precision_macro         : {m.get('precision_macro'):.6f}")
    print(f"recall_macro            : {m.get('recall_macro'):.6f}")

    if "balanced_accuracy_naive" in m:
        print("-" * len(title))
        print(f"balanced_accuracy_naive : {m.get('balanced_accuracy_naive'):.6f}")
        print(f"f1_macro_naive          : {m.get('f1_macro_naive'):.6f}")
        print(f"f1_weighted_naive       : {m.get('f1_weighted_naive'):.6f}")
        print(f"bal_acc_gain_vs_naive   : {m.get('balanced_accuracy_gain_vs_naive'):.6f}")
        print(f"f1_macro_gain_vs_naive  : {m.get('f1_macro_gain_vs_naive'):.6f}")


__all__ = [
    "ClassificationMetricsConfig",
    "compute_classification_metrics",
    "metrics_to_flat_dict",
    "print_classification_report_block",
]

Lo importamos así en las notebooks de entrenamiento:

```python
from metrics.classification_metrics import (
    compute_classification_metrics,
    metrics_to_flat_dict,
    print_classification_report_block,
)

result = compute_classification_metrics(
    y_true=y_valid,
    y_pred=y_pred_valid,
    model_name="lstm",
    split="valid",
    target="t2_dir_thr_90",
    labels=[-1, 0, 1],  # o las clases que estés usando
)

print_classification_report_block(result)

flat = metrics_to_flat_dict(result)
flat
```



##**Ejemplo de salida de `compute_classification_metrics`**

Supongamos:

```python
y_true = [1, 1, 0, -1, 1, 0, -1, 1]
y_pred = [1, 0, 0, -1, 1, 1, -1, 1]
```

Entonces:

```python
result = compute_classification_metrics(
    y_true=y_true,
    y_pred=y_pred,
    model_name="lstm",
    split="valid",
    target="t2_dir_thr_90",
    labels=[-1, 0, 1],
)
```

---

## **Output completo (`result`)**

```python
{
  'metrics': {
    'model': 'lstm',
    'split': 'valid',
    'target': 't2_dir_thr_90',
    'n_samples': 8,

    'accuracy': 0.75,
    'balanced_accuracy': 0.7777777777777778,

    'f1_macro': 0.7555555555555555,
    'f1_weighted': 0.75,

    'precision_macro': 0.7777777777777778,
    'precision_weighted': 0.7708333333333334,

    'recall_macro': 0.7777777777777778,
    'recall_weighted': 0.75,

    # baseline naive (clase más frecuente = 1)
    'accuracy_naive': 0.5,
    'balanced_accuracy_naive': 0.3333333333333333,
    'f1_macro_naive': 0.2222222222222222,
    'f1_weighted_naive': 0.3333333333333333,

    # mejora vs naive
    'balanced_accuracy_gain_vs_naive': 0.4444444444444445,
    'f1_macro_gain_vs_naive': 0.5333333333333333,
    'f1_weighted_gain_vs_naive': 0.4166666666666667
  },

  'confusion_matrix': [
    [2, 0, 0],  # real -1
    [0, 1, 1],  # real 0
    [0, 1, 3]   # real 1
  ],

  'confusion_matrix_labels': [-1, 0, 1],

  'class_distribution_true': {
    '-1': 2,
    '0': 2,
    '1': 4
  },

  'class_distribution_pred': {
    '-1': 2,
    '0': 2,
    '1': 4
  }
}
```

---

## **Output de `metrics_to_flat_dict(result)`**

Esto es lo que usarás para DataFrame:

```python
{
  'model': 'lstm',
  'split': 'valid',
  'target': 't2_dir_thr_90',
  'n_samples': 8,

  'accuracy': 0.75,
  'balanced_accuracy': 0.7777777777777778,

  'f1_macro': 0.7555555555555555,
  'f1_weighted': 0.75,

  'precision_macro': 0.7777777777777778,
  'precision_weighted': 0.7708333333333334,

  'recall_macro': 0.7777777777777778,
  'recall_weighted': 0.75,

  'accuracy_naive': 0.5,
  'balanced_accuracy_naive': 0.3333333333333333,

  'f1_macro_naive': 0.2222222222222222,
  'f1_weighted_naive': 0.3333333333333333,

  'balanced_accuracy_gain_vs_naive': 0.4444444444444445,
  'f1_macro_gain_vs_naive': 0.5333333333333333,
  'f1_weighted_gain_vs_naive': 0.4166666666666667
}
```

👉 Esto es lo que vas a usar para:

* comparar modelos
* hacer tablas tipo leaderboard
* guardar resultados

---

## **Output de `print_classification_report_block(result)`**

```text
=====================================================================
CLASSIFICATION REPORT | model=lstm | split=valid | target=t2_dir_thr_90
=====================================================================
n_samples               : 8
balanced_accuracy       : 0.777778
f1_macro                : 0.755556
f1_weighted             : 0.750000
accuracy                : 0.750000
precision_macro         : 0.777778
recall_macro            : 0.777778
---------------------------------------------------------------------
balanced_accuracy_naive : 0.333333
f1_macro_naive          : 0.222222
f1_weighted_naive       : 0.333333
bal_acc_gain_vs_naive   : 0.444444
f1_macro_gain_vs_naive  : 0.533333
```
